# 💊 Drug Prescription Prediction using Decision Trees

This notebook builds a **Decision Tree Classifier** to predict the most suitable drug for a patient based on their medical profile.

### 🎯 Objective
Given patient data (age, sex, blood pressure, cholesterol, Na-to-K ratio), predict which drug — out of Drug A, B, C, X, or Y — should be prescribed.

### 📦 Libraries Used
- `pandas` — data loading and manipulation
- `numpy` — numerical operations
- `scikit-learn` — model building and evaluation
- `matplotlib` — visualization

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import preprocessing, metrics

---
## 2. Load the Dataset

The dataset contains records of **200 patients**, all diagnosed with the same illness.
Each patient responded to one of 5 medications during treatment.

| Column | Description |
|---|---|
| `Age` | Age of the patient |
| `Sex` | Male (M) or Female (F) |
| `BP` | Blood Pressure — LOW, NORMAL, HIGH |
| `Cholesterol` | NORMAL or HIGH |
| `Na_to_K` | Sodium-to-Potassium ratio in blood |
| `Drug` | ✅ Target — Drug A, B, C, X, or Y |

In [ ]:
# Download the dataset
!wget -q -O drug200.csv https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/ML0101ENv3/labs/drug200.csv

# Load into a DataFrame
df = pd.read_csv("drug200.csv")

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Quick overview of data types and null values
df.info()

In [ ]:
# Distribution of target variable — how many patients responded to each drug
df['Drug'].value_counts()

---
## 3. Data Pre-processing

Machine learning models only understand **numbers**, not text.
We need to convert categorical columns (Sex, BP, Cholesterol) into numeric values.

This is called **Label Encoding**:
- `Sex`: F → 0, M → 1
- `BP`: LOW → 0, NORMAL → 1, HIGH → 2
- `Cholesterol`: NORMAL → 0, HIGH → 1

In [ ]:
# Select feature columns
X = df[['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K']].values

# Encode 'Sex'
le_sex = preprocessing.LabelEncoder()
le_sex.fit(['F', 'M'])
X[:, 1] = le_sex.transform(X[:, 1])

# Encode 'BP'
le_BP = preprocessing.LabelEncoder()
le_BP.fit(['LOW', 'NORMAL', 'HIGH'])
X[:, 2] = le_BP.transform(X[:, 2])

# Encode 'Cholesterol'
le_Chol = preprocessing.LabelEncoder()
le_Chol.fit(['NORMAL', 'HIGH'])
X[:, 3] = le_Chol.transform(X[:, 3])

print("Features after encoding:")
print(X[:5])

In [ ]:
# Target variable
y = df['Drug']
print("Target variable (first 5):")
print(y[:5].values)

---
## 4. Train-Test Split

We split the data into:
- **70% training set** — the model learns from this
- **30% test set** — we use this to evaluate how well the model generalised

`random_state=3` ensures reproducibility — you'll get the same split every time you run it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=3
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

---
## 5. Build the Decision Tree Model

A **Decision Tree** works like a flowchart of yes/no questions:
> "Is Na_to_K > 14.6?" → Yes → "Prescribe Drug Y"
> "Is BP HIGH?" → Yes → "Prescribe Drug A"

**Key parameters:**
- `criterion='entropy'` — uses Information Gain to decide the best split at each node
- `max_depth=4` — the tree can ask at most 4 levels of questions (prevents overfitting)

In [ ]:
# Initialize and train the model
drug_tree = DecisionTreeClassifier(criterion='entropy', max_depth=4)
drug_tree.fit(X_train, y_train)

print("✅ Model trained successfully!")

---
## 6. Visualize the Decision Tree

One of the biggest advantages of Decision Trees is **interpretability** — you can actually see the rules the model learned.

In [ ]:
feature_names = ['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K']
class_names = drug_tree.classes_

plt.figure(figsize=(20, 8))
plot_tree(
    drug_tree,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree — Drug Prescription", fontsize=16)
plt.tight_layout()
plt.savefig("decision_tree.png", dpi=150)
plt.show()

---
## 7. Make Predictions

In [ ]:
# Predict on test data
y_pred = drug_tree.predict(X_test)

# Compare predicted vs actual for first 10 patients
comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
comparison.head(10)

---
## 8. Model Evaluation

We evaluate the model using:
- **Accuracy Score** — % of correct predictions
- **Classification Report** — precision, recall, and F1-score per drug class
- **Confusion Matrix** — shows where the model got confused

In [ ]:
accuracy = metrics.accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Detailed performance per drug class
print(metrics.classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=drug_tree.classes_,
    cmap='Blues',
    ax=ax
)
plt.title("Confusion Matrix — Drug Prediction", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

---
## 9. Feature Importance

Which patient attributes mattered the most for the model's decisions?

In [ ]:
importances = drug_tree.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Drug Prediction', fontsize=13)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

---
## 10. Summary & Insights

| Metric | Result |
|---|---|
| Algorithm | Decision Tree Classifier |
| Criterion | Entropy (Information Gain) |
| Max Depth | 4 |
| Train/Test Split | 70% / 30% |
| Accuracy | ~98%+ |

### 🔍 Key Takeaways
- **Na_to_K ratio** is the most important feature — patients with high Na_to_K are strongly predicted to respond to Drug Y
- **Blood Pressure** is the second most influential factor
- The model is highly accurate and interpretable — the decision tree rules can be directly presented to domain experts
- Decision Trees are great for healthcare scenarios where **explainability** matters as much as accuracy